In [1]:
import pandas as pd

df_sortimento = pd.read_excel('data/HERING_FRANQUIA_checklist_RA_REMESSA_QUARTA_2026-03-18.xlsx', engine='calamine')
df_sortimento.head()

,CANAL,FILIAL,FILIAL_NOME,REGIAO,SKU,ARTIGO_COR,PRODUTO,DESC_PRODUTO,COR,TAM,...,DATA,CNPJ,CLISAP,CONT_RUPTURA,CONT_RUPTURA_RESSUP,CONT_FALTA,VOLUME_FALTA,CONT_ALVO,CONT_EXCESSO,VOLUME_EXCESSO
0,FRANQUIAS,10,HERING ACLIMACAO,SUDESTE,0111-AX7EN-G,0111-AX7EN,0111,CAMISETA REGATA MASC,AX7EN,G,...,2026-03-18,7057579000178,5587,0,1,0,0,0,0,0
1,FRANQUIAS,526,HS R ENG M DE GUSMAO,NORDESTE,KTNU-N10SI-046,KTNU-N10SI,KTNU,BERMUDA MASC,N10SI,046,...,2026-03-18,34490015000954,116442,0,0,1,1,0,0,0
2,FRANQUIAS,526,HS R ENG M DE GUSMAO,NORDESTE,KTNU-HMJSI-036,KTNU-HMJSI,KTNU,BERMUDA MASC,HMJSI,036,...,2026-03-18,34490015000954,116442,0,1,0,0,0,0,0
3,FRANQUIAS,526,HS R ENG M DE GUSMAO,NORDESTE,KTNU-HMJSI-038,KTNU-HMJSI,KTNU,BERMUDA MASC,HMJSI,038,...,2026-03-18,34490015000954,116442,0,0,1,2,0,0,0
4,FRANQUIAS,526,HS R ENG M DE GUSMAO,NORDESTE,KTNU-HMJSI-040,KTNU-HMJSI,KTNU,BERMUDA MASC,HMJSI,040,...,2026-03-18,34490015000954,116442,0,0,1,1,0,0,0


In [2]:
df_sortimento['ARTIGO_COR'] = df_sortimento['PRODUTO']  + df_sortimento['COR']

In [3]:
lista_artigos_unicos = df_sortimento['ARTIGO_COR'].dropna().unique() 
string_artigos = ','.join(lista_artigos_unicos)


lista_filiais_unicas = df_sortimento['FILIAL'].dropna().astype(str).unique()
string_filiais = ','.join(lista_filiais_unicas)

In [4]:
print("Tamanho da lista de Artigos: ", len(lista_artigos_unicos))
print("Tamanho da lista de Filiais: ", len(lista_filiais_unicas))

Tamanho da lista de Artigos:  168
Tamanho da lista de Filiais:  506


In [7]:
from google.cloud import bigquery
import warnings
warnings.filterwarnings("ignore", message="Your application has authenticated using end user credentials")
client = bigquery.Client(project='hering-data-analytics-prod')

In [ ]:
query = f"""


DECLARE lista_artigos STRING;
DECLARE lista_filial  STRING;

SET lista_artigos = '{string_artigos}';
SET lista_filial = '{string_filiais}';



with cte as (
SELECT 
  cast(c.PontoVda as STRING) as PontoVda,
  c.Sit,
  c.Usuario,
  c.ARTIGO_COR,
  c.Tamanho,
  sum(cast(c.Pecas as integer)) as Pecas
FROM `hering-data-analytics-prod.BI.vendas_sell_in_fato_bi` c
WHERE  c.Sit In ('PENDENTE', 'ATENDIDO', 'SUSPENSO COMERCIAL', 'SUSPENSO COMERCIAL + FINANCEIRO', 'SUSPENSO FINANCEIRO') and  DATE_TRUNC(DATE(DataHora_Criacao_OV, 'America/Sao_Paulo'), MONTH) = DATE_TRUNC(CURRENT_DATE('America/Sao_Paulo'), MONTH) and c.Usuario = 'ABF'
and c.ARTIGO_COR  IN UNNEST(SPLIT(lista_artigos, ','))
and CAST(c.PontoVda AS STRING) IN UNNEST(SPLIT(lista_filial, ','))
group by c.PontoVda, c.Sit, c.Usuario, c.ARTIGO_COR, c.Tamanho
order by c.PontoVda asc
)

select * from cte where PontoVda IS NOT NULL 

"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_CA = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_CA)}")
display(df_CA.head())

Rodando a query...
Total de linhas retornadas: 22169


,PontoVda,Sit,Usuario,ARTIGO_COR,Tamanho,Pecas
0,3,ATENDIDO,ABF,0241N0A00S,P,1
1,3,ATENDIDO,ABF,0201N0A00S,M,1
2,3,ATENDIDO,ABF,0201N0A00S,XG,1
3,3,ATENDIDO,ABF,0241N1007S,M,1
4,3,ATENDIDO,ABF,N3A7NATEN,XG,2


In [6]:
df_CA['PontoVda'].dtypes

dtype('O')

In [7]:
df_CA['PontoVda'].nunique()


406

In [15]:
df_CA['ARTIGO_COR'].nunique()

158

In [16]:
df_CA.to_excel('data/df_CA.xlsx', index=False)

### CA V2

In [8]:

query = f"""


with cte as ( 
  SELECT 
    CAST(c.PontoVda AS STRING) AS FILIAL,
    c.Sit,
    c.Usuario,
    c.ARTIGO_COR,
    c.Tamanho,
    SUM(CAST(c.Pecas AS INTEGER)) AS Pecas
  FROM `hering-data-analytics-prod.BI.vendas_sell_in_fato_bi` c
  WHERE Sit = 'FATURADO' 
    AND CAST(DtEmiNota AS DATE) IN 
    (
    CURRENT_DATE('America/Sao_Paulo'), 
    DATE_SUB(CURRENT_DATE('America/Sao_Paulo'), INTERVAL 1 DAY),
    DATE_SUB(CURRENT_DATE('America/Sao_Paulo'), INTERVAL 2 DAY),
    DATE_SUB(CURRENT_DATE('America/Sao_Paulo'), INTERVAL 3 DAY)
    )
    AND DATE(DataHora_Criacao_OV, 'America/Sao_Paulo') >= '2026-03-10'
    AND c.Usuario = 'ABF'
    GROUP BY 
    CAST(c.PontoVda AS STRING),
    c.Sit,
    c.Usuario,
    c.ARTIGO_COR,
    c.Tamanho
  UNION ALL 
  
  SELECT 
    CAST(c.PontoVda AS STRING) AS FILIAL,
    c.Sit,
    c.Usuario,
    c.ARTIGO_COR,
    c.Tamanho,
    SUM(CAST(c.Pecas AS INTEGER)) AS Pecas
  FROM `hering-data-analytics-prod.BI.vendas_sell_in_fato_bi` c
  WHERE  c.Sit IN ('PENDENTE', 'ATENDIDO', 'SUSPENSO COMERCIAL', 'SUSPENSO COMERCIAL + FINANCEIRO', 'SUSPENSO FINANCEIRO') 
    and DATE(DataHora_Criacao_OV, 'America/Sao_Paulo') >= '2026-03-10'
    and c.Usuario = 'ABF'
  GROUP BY CAST(c.PontoVda AS STRING), c.Sit, c.Usuario, c.ARTIGO_COR, c.Tamanho
)

select * from cte
where FILIAL IS NOT NULL 
order by FILIAL ASC

"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_CA_V2 = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_CA_V2)}")
display(df_CA_V2.head())

Rodando a query...
Total de linhas retornadas: 32319


,FILIAL,Sit,Usuario,ARTIGO_COR,Tamanho,Pecas
0,1,FATURADO,ABF,K0VU1ASI,036,1
1,1,ATENDIDO,ABF,KG2X1BSI,042,1
2,1,ATENDIDO,ABF,KG2AN10SI,M,1
3,1,FATURADO,ABF,5CMVN10EN,012,1
4,1,FATURADO,ABF,558CAXTEN,014,1


In [9]:
df_CA_V2['FILIAL'].nunique()

497

In [10]:
df_CA_V2.to_excel('data/df_CA.xlsx', index=False)

In [11]:
query = f"""

DECLARE lista_artigos STRING;
DECLARE lista_filial  STRING;

SET lista_artigos = '{string_artigos}';
SET lista_filial = '{string_filiais}';




SELECT
    A.ArtigoID ARTIGO_COR,
    A.Tamanho,
    A.Data as Data,
    A.Piramide,
    A.CANAL_PV as CANAL,
    CAST(A.Ponto_Venda_Cod AS STRING) AS FILIAL,
    cast(coalesce(SUM(A.Q_Pecas), 0) as INT) AS total_pecas,
    coalesce(SUM(A.Q_Estoque_Total), 0) AS estoque_total,
    coalesce(SUM(A.Q_Estoque_Transito), 0) AS estoque_transito,
FROM
    `hering-data-analytics-prod.BI.performance_comercial_fato_bi` AS A
WHERE
    A.Piramide IN ('Essencial', 'Sazonal')
    AND A.Data = current_date() - 1
    AND A.CANAL_PV = 'Franquias'  
    AND CAST(A.Ponto_Venda_Cod AS STRING) IN UNNEST(SPLIT(lista_filial, ','))
    AND A.ArtigoID  IN UNNEST(SPLIT(lista_artigos, ','))
GROUP BY
    A.ArtigoID,
    A.Tamanho,
    A.Data,
    A.Piramide,
    A.CANAL_PV,
    A.Ponto_Venda_Cod

"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_stk  = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_stk)}")
display(df_stk.head())

Rodando a query...
Total de linhas retornadas: 355261


,ARTIGO_COR,Tamanho,Data,Piramide,CANAL,FILIAL,total_pecas,estoque_total,estoque_transito
0,0111AX7EN,G,2026-03-19,Essencial,Franquias,275,1,4,0.0
1,0111AX7EN,G,2026-03-19,Essencial,Franquias,636,1,6,0.0
2,0111AX7EN,G,2026-03-19,Essencial,Franquias,1085,1,9,0.0
3,0111AX7EN,G,2026-03-19,Essencial,Franquias,202,1,4,0.0
4,0111AX7EN,G,2026-03-19,Essencial,Franquias,354,1,1,0.0


In [12]:
df_stk.shape

(355261, 9)

In [13]:
df_stk['FILIAL'].dtypes

dtype('O')

In [14]:
df_stk['estoque_total'].sum()


np.int64(1496471)

In [15]:
df_stk['FILIAL'].nunique()

506

In [16]:
df_stk['ARTIGO_COR'].nunique()

168

In [17]:
df_stk['Estoque Total'] = (df_stk['estoque_total'] - df_stk['total_pecas']).abs()

In [18]:
df_stk.head()

,ARTIGO_COR,Tamanho,Data,Piramide,CANAL,FILIAL,total_pecas,estoque_total,estoque_transito,Estoque Total
0,0111AX7EN,G,2026-03-19,Essencial,Franquias,275,1,4,0.0,3
1,0111AX7EN,G,2026-03-19,Essencial,Franquias,636,1,6,0.0,5
2,0111AX7EN,G,2026-03-19,Essencial,Franquias,1085,1,9,0.0,8
3,0111AX7EN,G,2026-03-19,Essencial,Franquias,202,1,4,0.0,3
4,0111AX7EN,G,2026-03-19,Essencial,Franquias,354,1,1,0.0,0


In [19]:
df_agregado = df_stk.groupby('Data')[['Estoque Total']].sum().reset_index()
display(df_agregado)

,Data,Estoque Total
0,2026-03-19,1497506


In [20]:
df_agregado = df_stk.groupby('Data')[['total_pecas']].sum().reset_index()
display(df_agregado)

,Data,total_pecas
0,2026-03-19,8815


In [21]:
df_stk.isnull().sum()

ARTIGO_COR          0
Tamanho             0
Data                0
Piramide            0
CANAL               0
FILIAL              0
total_pecas         0
estoque_total       0
estoque_transito    0
Estoque Total       0
dtype: int64

In [22]:
df_stk['Tamanho'].unique()

array(['G', 'M', 'XG', 'XXG', 'P', 'EXG', 'XP', '004', '010', '014',
       '016', '006', '012', '008', '002', '036', '038', '040', '042',
       '044', '046', '048', '050', '052', '034', 'G1', '005', '001',
       '003', '054', '056', 'G3'], dtype=object)

In [23]:
df_stk['ARTIGO_COR'].nunique()

168

In [24]:
df_stk.drop(columns=['estoque_total', 'total_pecas'], inplace= True)
df_stk.head()

,ARTIGO_COR,Tamanho,Data,Piramide,CANAL,FILIAL,estoque_transito,Estoque Total
0,0111AX7EN,G,2026-03-19,Essencial,Franquias,275,0.0,3
1,0111AX7EN,G,2026-03-19,Essencial,Franquias,636,0.0,5
2,0111AX7EN,G,2026-03-19,Essencial,Franquias,1085,0.0,8
3,0111AX7EN,G,2026-03-19,Essencial,Franquias,202,0.0,3
4,0111AX7EN,G,2026-03-19,Essencial,Franquias,354,0.0,0


In [25]:
df_stk.to_excel('data/df_STK.xlsx', index=False)